# 10 — Caption Generation

**Marker:** `NOTEBOOK_10_CAPTION_GENERATION_FRESH_V1`

This notebook builds synchronized captions from Notebook 09's exact segment
durations and TTS text.

It produces:

- approximate word-level timing data
- readable phrase-based caption cues
- SRT
- WebVTT
- styled ASS captions for vertical-video burn-in
- a caption manifest for Notebook 11

No speech-recognition model is required.


## Load the project

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.captions import (
    find_tts_manifest,
    generate_caption_assets,
    load_tts_manifest,
    summarize_caption_manifest,
)

print("NOTEBOOK_10_CAPTION_GENERATION_FRESH_V1")
print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
AUDIO_DIRECTORY = PROJECT_ROOT / "data" / "audio"
CAPTION_OUTPUT_DIRECTORY = PROJECT_ROOT / "data" / "captions"

# Leave as None to use the newest tts_manifest.json found recursively.
TTS_MANIFEST_FILENAME = None

CAPTION_STYLE = "phrase"
MAX_WORDS_PER_CUE = 5
MAX_CHARACTERS_PER_LINE = 22
MAX_LINES = 2
MINIMUM_CUE_SECONDS = 0.55
MAXIMUM_CUE_SECONDS = 2.6

# Burn-in appearance used by the generated ASS file.
ASS_FONT_NAME = "Arial"
ASS_FONT_SIZE = 72
ASS_MARGIN_V = 300

print(f"Audio input: {AUDIO_DIRECTORY}")
print(f"Caption output: {CAPTION_OUTPUT_DIRECTORY}")
print(f"Caption style: {CAPTION_STYLE}")

## Load the newest TTS manifest

In [ ]:
tts_manifest_path = find_tts_manifest(
    audio_directory=AUDIO_DIRECTORY,
    filename=TTS_MANIFEST_FILENAME,
)
tts_manifest = load_tts_manifest(tts_manifest_path)

print(f"TTS manifest: {tts_manifest_path}")
print(f"Topic: {tts_manifest.source_topic_title}")
print(f"Voice: {tts_manifest.voice}")
print(f"Segments: {len(tts_manifest.segments)}")
print(f"Audio duration: {tts_manifest.actual_duration_seconds}s")

## Inspect the exact TTS text

In [ ]:
for segment in tts_manifest.segments:
    print(
        f"{segment.index:02d} — {segment.segment_type} "
        f"({segment.actual_seconds:.2f}s)"
    )
    print(segment.tts_text)
    print()

## Generate all caption formats

In [ ]:
caption_manifest = generate_caption_assets(
    tts_manifest=tts_manifest,
    source_tts_manifest_path=tts_manifest_path,
    output_root=CAPTION_OUTPUT_DIRECTORY,
    caption_style=CAPTION_STYLE,
    max_words_per_cue=MAX_WORDS_PER_CUE,
    max_characters_per_line=MAX_CHARACTERS_PER_LINE,
    max_lines=MAX_LINES,
    minimum_cue_seconds=MINIMUM_CUE_SECONDS,
    maximum_cue_seconds=MAXIMUM_CUE_SECONDS,
    ass_font_name=ASS_FONT_NAME,
    ass_font_size=ASS_FONT_SIZE,
    ass_margin_v=ASS_MARGIN_V,
)

for name, value in summarize_caption_manifest(
    caption_manifest
).items():
    print(f"{name}: {value}")

## Preview caption cues

In [ ]:
for cue in caption_manifest.cues:
    print(
        f"{cue.cue_index:03d} "
        f"{cue.start_seconds:6.2f} --> {cue.end_seconds:6.2f} "
        f"[{cue.segment_type}]"
    )
    print(cue.text)
    print()

## Basic timing checks

In [ ]:
audio_end = caption_manifest.audio_duration_seconds
caption_end = caption_manifest.final_caption_end_seconds
difference = caption_end - audio_end

print(f"Audio duration: {audio_end:.3f}s")
print(f"Final caption end: {caption_end:.3f}s")
print(f"Difference: {difference:+.3f}s")

overlaps = []

for current, following in zip(
    caption_manifest.cues,
    caption_manifest.cues[1:],
):
    if current.end_seconds > following.start_seconds:
        overlaps.append(
            (current.cue_index, following.cue_index)
        )

print(f"Overlapping cue pairs: {len(overlaps)}")

if abs(difference) > 1.0:
    print(
        "WARNING: Captions end more than one second from the audio end."
    )
else:
    print("Caption ending is close to the narration ending.")

if overlaps:
    print(f"WARNING: overlaps found: {overlaps[:10]}")
else:
    print("No caption overlaps detected.")

## Preview generated SRT

In [ ]:
caption_directory = Path(
    caption_manifest.output_directory
)
srt_path = caption_directory / caption_manifest.srt_filename

srt_text = srt_path.read_text(encoding="utf-8")
print(srt_text[:4000])

## Inspect output files

In [ ]:
for path in sorted(caption_directory.iterdir()):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

## Preview complete caption manifest

In [ ]:
print(caption_manifest.model_dump_json(indent=2))